### Forward simulation: covariate drifts and experimental design

For accumulator $i$ on trial $r$, this example uses $\mu_{i,r}=\beta_{i,0}+z_r\beta_{i,1}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cssm import race_multistage

RANDOM_STATE = 20260820
OMISSION = -999.0
dt = 1e-3
T = 1.5

# Three experimental conditions: evidence favours option 1, neither, option 0.
z = np.array([-1.0, 0.0, 1.0])
beta_intercept = np.array([0.70, 0.70])
beta_covariate = np.array([0.30, -0.30])
mu_by_trial = beta_intercept[None, :] + z[:, None] * beta_covariate[None, :]
mu_by_trial

In [ ]:
n_trials = z.size
design_arrays = {
    'mu_array': mu_by_trial[:, :, None],
    'sigma_array': np.ones((n_trials, 2, 1)),
    'node_array': np.zeros((n_trials, 2, 1)),
    'd_array': np.ones((n_trials, 2), dtype=np.int32),
    'upper_intercept_array': np.ones((n_trials, 2, 1)),
    'upper_slope_array': np.zeros((n_trials, 2, 1)),
    'x0_array': np.zeros((n_trials, 2)),
}

out = race_multistage(
    **design_arrays, n_samples=20_000, delta_t=dt, max_t=T, random_state=RANDOM_STATE,
)
rts = out['rts'][:, :, 0]
choices = out['choices'][:, :, 0]
responded = rts != OMISSION

choice_probability = np.array([
    [np.mean(responded[:, r] & (choices[:, r] == i)) for i in range(2)]
    for r in range(n_trials)
])
mean_rt = np.array([rts[responded[:, r], r].mean() for r in range(n_trials)])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(z, choice_probability[:, 0], 'o-', label='choice 0')
ax[0].plot(z, choice_probability[:, 1], 'o-', label='choice 1')
ax[0].set(xlabel=r'covariate $z$', ylabel='unconditional choice probability', ylim=(0, 1))
ax[0].legend()

ax[1].plot(z, mean_rt, 'o-', color='tab:purple')
ax[1].set(xlabel=r'covariate $z$', ylabel='mean RT among responses')
plt.show()

for r in range(n_trials):
    print(f'z={z[r]:+.0f}: mu={mu_by_trial[r]}, P(choice 0)={choice_probability[r, 0]:.3f}, P(choice 1)={choice_probability[r, 1]:.3f}, mean RT={mean_rt[r]:.3f}')